# 04 — Insights
Simple L/S strategy backtest on up-shock events, equity curve, summary tables.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from event_study import OIL_SHOCK_EVENTS, get_sector_returns
from utils import DATA_PROC, TABLES_DIR, PLOTS_DIR, set_theme, save_table
from constants import MARKET_COL, OIL_COL, SECTOR_MAP


In [2]:
returns_raw = pd.read_parquet(DATA_PROC / 'returns.parquet')
returns     = get_sector_returns(returns_raw)
print('Sector columns:', [c for c in returns.columns
                          if c not in (MARKET_COL, OIL_COL, 'USDINR')])


Sector columns: ['OIL_GAS', 'AUTO', 'FMCG', 'IT', 'PHARMA']


In [3]:
# Simple long OIL_GAS / short (AUTO+FMCG)/2 on up-shock events
HOLD_DAYS  = 3
UP_EVENTS  = [e['date'] for e in OIL_SHOCK_EVENTS if e['direction'] == 'up']

strat_rows = []
for date_str in UP_EVENTS:
    event_dt = pd.Timestamp(date_str)
    idx = returns.index.searchsorted(event_dt)
    if idx + HOLD_DAYS >= len(returns):
        continue
    hold = returns.iloc[idx + 1 : idx + 1 + HOLD_DAYS]
    leg_long  =  hold['OIL_GAS'].sum()
    leg_short = -(hold['AUTO'].sum() + hold['FMCG'].sum()) / 2
    pnl       = (leg_long + leg_short) / 2
    strat_rows.append({'date': date_str, 'pnl': pnl})

strat_df = pd.DataFrame(strat_rows)
print(strat_df.to_string(index=False))
print(f'\nMean PnL / trade : {strat_df.pnl.mean()*100:.2f}%')
print(f'Win rate          : {(strat_df.pnl > 0).mean()*100:.0f}%')
print(f'Total (compounded): {((1+strat_df.pnl).prod()-1)*100:.2f}%')


      date       pnl
2019-09-14  0.003481
2021-03-08  0.006264
2022-02-24 -0.009339
2023-04-02  0.014445
2023-10-07  0.001443
2024-01-12 -0.016858
2024-04-14 -0.004858

Mean PnL / trade : -0.08%
Win rate          : 57%
Total (compounded): -0.57%


In [4]:
# Equity curve
set_theme()
strat_curve  = (1 + strat_df['pnl']).cumprod()
nifty_subset = returns_raw.loc[
    returns_raw.index.isin([pd.Timestamp(d) for d in strat_df['date']])
][MARKET_COL]
nifty_curve  = (1 + nifty_subset).cumprod().values

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(len(strat_curve)), strat_curve.values,
        label='Strategy (Long OIL_GAS / Short AUTO+FMCG)', lw=2)
ax.plot(range(len(nifty_curve)), nifty_curve,
        label='NIFTY on event days', lw=1.5, ls='--')
ax.axhline(1.0, color='grey', lw=0.8, ls=':')
ax.set_xlabel('Trade #'); ax.set_ylabel('Equity (start=1)')
ax.set_title('Strategy equity curve — up-shock events')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'strategy_equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()


In [5]:
# Read significance & asymmetry tables produced by NB02
sig  = pd.read_csv(TABLES_DIR / 'significance_test.csv')
asym = pd.read_csv(TABLES_DIR / 'asymmetry_test.csv')

print('=== SIGNIFICANCE (10-event average) ===')
print(sig[['sector','mean_car_pct','t_stat','p_value','significant']].to_string(index=False))
print()
print('=== ASYMMETRY (up-shock vs down-shock) ===')
# Column names match asymmetry_test() output exactly
print(asym[['sector','mean_car_up_pct','mean_car_down_pct','t_stat','p_value','asymmetric']].to_string(index=False))
print('NB04 complete ✓')


=== SIGNIFICANCE (10-event average) ===
 sector  mean_car_pct  t_stat  p_value  significant
OIL_GAS         0.137   0.093   0.9278        False
   AUTO        -0.132  -0.069   0.9467        False
   FMCG         0.157   0.109   0.9156        False
     IT        -0.220  -0.137   0.8940        False
 PHARMA        -2.015  -0.985   0.3504        False

=== ASYMMETRY (up-shock vs down-shock) ===
 sector  mean_car_up_pct  mean_car_down_pct  t_stat  p_value  asymmetric
OIL_GAS           -1.048              2.902  -1.173   0.3191       False
   AUTO           -0.793              1.412  -0.344   0.7605       False
   FMCG           -2.048              5.300  -2.901   0.0703        True
     IT           -0.213             -0.237   0.004   0.9968       False
 PHARMA           -0.760             -4.944   0.745   0.5172       False
NB04 complete ✓
